# Script 2 — Interactive HTML Table Generation

This notebook reads the master catalogue produced by Script 1 and renders it as an interactive HTML table with formatted values, asymmetric uncertainties, colour-coded references, in-cell habitability-zone badges, and DataTables search and sorting.

**Author:** Rafael Carmona Vendoiro, UCM  
**Last updated:** 22/04/2026  
**Input:** `Exoplanets_SolarNeighbourhood_Catalogue.csv`  
**Output:** `exoplanets_catalogue.html`

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import re
import urllib.parse

## File Configuration

In [2]:
table_title = "Exoplanet Catalogue in the Solar Neighbourhood (d < 10 pc)"
file_input  = Path('Exoplanets_SolarNeighbourhood_Catalogue.csv')
file_output = Path('exoplanets_catalogue.html')

## Step 1 — Helper Functions

### `format_val_and_error`
Formats a physical value and its uncertainties to the appropriate number of decimal places.
Handles string values such as upper/lower limits (`<`, `>`), symmetric vs asymmetric errors,
and missing data.

### `format_reference_smart`
Assigns a colour-coded HTML style to each reference string based on its source catalogue
(CIF25, Exoplanet.eu, NASA, Gaia) or, for individual paper links, based on publication year.

In [3]:
def format_val_and_error(val, err_min, err_max):
    """
    Format a value and its uncertainties for HTML display.

    Returns (val_str, err_str). If the value is empty or a limit string
    (e.g. '<0.5'), returns it as-is with no error string.
    """
    if pd.isna(val):
        return "", ""

    # Handle string values such as upper/lower limits (<, >, ~)
    if isinstance(val, str):
        val_clean = val.strip()
        try:
            val = float(val_clean)
        except ValueError:
            return val_clean, ""

    if pd.isna(err_min) and pd.isna(err_max):
        return f"{val:.4g}", ""

    e_min = abs(err_min) if pd.notna(err_min) else np.nan
    e_max = abs(err_max) if pd.notna(err_max) else np.nan
    if pd.isna(e_min): e_min = e_max
    if pd.isna(e_max): e_max = e_min

    if pd.isna(e_max) or e_max == 0 or np.isinf(e_max):
        return f"{val:.4g}", ""

    def get_needed_decimals(x):
        if pd.isna(x) or x == 0 or np.isinf(x):
            return 2
        try:
            order_of_mag = np.floor(np.log10(x))
            decs = int(2 - 1 - order_of_mag)
            return max(0, min(9, decs))
        except Exception:
            return 2

    decimals = max(get_needed_decimals(e_max), get_needed_decimals(e_min))

    try:
        mean_err = (e_max + e_min) / 2
        if mean_err == 0:
            is_symmetric, err_use = True, 0
        elif (abs(e_max - e_min) / mean_err) < 0.05:
            is_symmetric, err_use = True, mean_err
        else:
            is_symmetric, err_use = False, max(e_max, e_min)
    except Exception:
        is_symmetric, err_use = True, e_max

    def fmt(n, dec):
        return f"{n:.{dec}f}"

    val_str = fmt(val, decimals)

    if is_symmetric:
        err_str = f"&#177; {fmt(err_use, decimals)}"
    else:
        up  = fmt(e_max, decimals)
        lo  = fmt(e_min, decimals)
        err_str = (
            "<div style='display:inline-block; vertical-align:middle; "
            "line-height:1; font-size:0.8em;'>"
            f"<span style='display:block;'>+{up}</span>"
            f"<span style='display:block;'>-{lo}</span></div>"
        )

    return val_str, err_str


def format_hz_zone_badge(zone):
    """Return a compact semantic badge for the habitable-zone label."""
    zone_str = str(zone).strip()
    if not zone_str or zone_str.lower() == 'nan':
        zone_str = 'No data'

    palette = {
        'Conservative HZ': ('#e8f5e9', '#1b5e20'),
        'Optimistic HZ (inner)': ('#fff4db', '#8a5a00'),
        'Optimistic HZ (outer)': ('#fff0e0', '#9a3412'),
        'Too hot': ('#fde8e8', '#991b1b'),
        'Too cold': ('#e8f1fb', '#1e3a8a'),
        'No data': ('#f1f3f5', '#495057'),
    }

    bg, fg = palette.get(zone_str, ('#f1f3f5', '#495057'))
    return (
        f"<span style='display:inline-block; padding:3px 8px; border-radius:999px; "
        f"background-color:{bg}; color:{fg}; font-weight:700; font-size:0.82em; "
        f"line-height:1.2; white-space:nowrap;'>{zone_str}</span>"
    )


def format_planet_category_badge(category):
    """Return a compact semantic badge for the planet size category."""
    cat = str(category).strip()
    if not cat or cat.lower() == 'nan':
        cat = 'Unknown'

    # Solar-system-inspired palette with high readability.
    palette = {
        'Terrestrial': ('#f3e3cf', '#5a3a1b'),   # Mercury/Venus rocky tone
        'Sub-Neptune': ('#dff2ff', '#0b4f6c'),   # Uranus-like cyan tone
        'Ice Giant': ('#d6ecff', '#1e3a8a'),     # Neptune deep blue tone
        'Gas Giant': ("#ffd697", "#a04d11"),     # Jupiter/Saturn warm tone
        'Unknown': ('#f1f3f5', '#495057'),
    }

    bg, fg = palette.get(cat, ('#f1f3f5', '#495057'))
    return (
        f"<span style='display:inline-block; padding:3px 8px; border-radius:999px; "
        f"background-color:{bg}; color:{fg}; font-weight:700; font-size:0.82em; "
        f"line-height:1.2; white-space:nowrap;'>{cat}</span>"
    )


def format_reference_smart(ref_html):
    """
    Apply colour-coded HTML styling to a reference string.

    Colour scheme:
      - Exoplanet.eu  -> pink  (#e83e8c)
      - NASA / IPAC   -> orange (#fd7e14)
      - CIF25         -> blue  (#007bff)
      - Gaia / VizieR -> green (#28a745)
      - Papers by year: <2010 grey, 2010-15 muted blue, 2016-20 deep blue,
                        2021-23 violet, 2024+ dark indigo
      - TO-DO flag    -> red alert badge
    """
    if pd.isna(ref_html) or not str(ref_html).strip():
        return ""

    ref_str   = str(ref_html).strip()
    ref_lower = ref_str.lower()

    # Flag any unresolved references for review
    if "todo" in ref_lower or "to-do" in ref_lower:
        return (
            "<span style='color:#721c24; background-color:#f8d7da; padding:2px 5px; "
            "border-radius:4px; font-weight:bold; border:1px solid #f5c6cb;'>TO-DO</span>"
        )

    text_only = re.sub(r'<[^>]+>', '', ref_lower).strip()

    # Assign base colour for known catalogues
    color_base = None
    if text_only == 'eu' or 'exoplanet.eu' in ref_lower:
        color_base = "#e83e8c"
    elif 'nasa' in ref_lower or 'ipac' in ref_lower:
        color_base = "#fd7e14"
    elif 'cif' in text_only or 'cif25' in text_only:
        color_base = "#007bff"
    elif any(x in ref_lower for x in ('gaia', 'dr2', 'dr3', 'dr4', 'vizier')):
        color_base = "#28a745"

    if color_base:
        if "<a " in ref_str:
            ref_str = re.sub(r'style="[^"]*"', '', ref_str)
            return ref_str.replace(
                "<a ",
                f"<a style='text-decoration:none; font-weight:bold; color:{color_base};' "
            )
        else:
            return f"<span style='color:{color_base}; font-weight:bold;'>{ref_str}</span>"

    # For individual paper links: colour by publication year
    if "<a " in ref_str:
        match      = re.search(r'\b(199\d|20[0-2]\d)\b', ref_str)
        color_year = "#6f42c1"  # default: mid-purple

        if match:
            year = int(match.group(1))
            if year < 2010:
                color_year = "#6c757d"   # grey (older papers)
            elif year <= 2015:
                color_year = "#5c6bc0"   # muted blue
            elif year <= 2020:
                color_year = "#3949ab"   # deep blue
            elif year <= 2023:
                color_year = "#8e24aa"   # vibrant violet
            else:
                color_year = "#4a148c"   # dark indigo (2024-2025)

        ref_str = re.sub(r'style="[^"]*"', '', ref_str)
        return ref_str.replace(
            "<a ",
            f"<a style='text-decoration:none; font-weight:bold; color:{color_year};' "
        )

    # Fallback: plain text with no recognised pattern
    return f"<span style='color:#6c757d; font-weight:bold;'>{ref_str}</span>"

## Step 2 — Data Assembly

This step builds the row-level data used by the final table. Each block is assembled into `internal_data` and later combined into the MultiIndex output.

The assembled content includes the SIMBAD-linked planet name, discovery and detection metadata, habitability-zone and planet-category badges, physical parameters with uncertainties and references, coordinates, spectral type, and external database links.

In [4]:
print(f"Reading master catalogue: {file_input}...")
df = pd.read_csv(file_input)

# Physical parameter columns to include, with their CSV source names.
# The HZ zone badge and planet category badge are handled separately because they are not numeric quantities.
params = {
    'd':          {'unit': 'pc',                       'val': 'star_distance',   'min': 'star_distance_error_min',  'max': 'star_distance_error_max'},
    'P_orb':      {'unit': 'd',                        'val': 'orbital_period',  'min': 'orbital_period_error_min', 'max': 'orbital_period_error_max'},
    'a_p':        {'unit': 'au',                       'val': 'semi_major_axis', 'min': 'semi_major_axis_error_min','max': 'semi_major_axis_error_max'},
    'e_p':        {'unit': '',                         'val': 'eccentricity',    'min': 'eccentricity_error_min',   'max': 'eccentricity_error_max'},
    'i_p':        {'unit': 'deg',                      'val': 'inclination',     'min': 'inclination_error_min',    'max': 'inclination_error_max'},
    'T_0,p':      {'unit': 'BJD',                      'val': 'T0_val',          'min': 'T0_min',                   'max': 'T0_max'},
    '&#969;_p':   {'unit': 'deg',                      'val': 'omega',           'min': 'omega_error_min',          'max': 'omega_error_max'},
    'R_p':        {'unit': 'R<sub>&#8853;</sub>',      'val': 'radius_earth_th', 'min': 'radius_earth_error_min',   'max': 'radius_earth_error_max',   'ref': 'radius_ref'},
    'M_p':        {'unit': 'M<sub>&#8853;</sub>',      'val': 'mass_earth',      'min': 'mass_earth_error_min',     'max': 'mass_earth_error_max',     'ref': 'mass_ref'},
    'T_eq':       {'unit': 'K',                        'val': 'Teq_val',         'min': 'Teq_error_min',            'max': 'Teq_error_max'},
    'R_star':     {'unit': 'R<sub>&#9737;</sub>',      'val': 'star_radius',     'min': 'star_radius_error_min',    'max': 'star_radius_error_max'},
    'M_star':     {'unit': 'M<sub>&#9737;</sub>',      'val': 'star_mass',       'min': 'star_mass_error_min',      'max': 'star_mass_error_max'},
    '&#961;_star':{'unit': 'g cm<sup>-3</sup>',        'val': 'star_density',    'min': 'star_density_error_min',   'max': 'star_density_error_max'},
    'T_eff':      {'unit': 'K',                        'val': 'star_teff',       'min': 'star_teff_error_min',      'max': 'star_teff_error_max'},
}

internal_data = {}

# --- Planet ID column: name linked to SIMBAD ---
planet_links = []
for planet_name in df['name']:
    planet_name  = str(planet_name).strip()
    encoded_name = urllib.parse.quote(planet_name)
    url_simbad   = f"https://simbad.cds.unistra.fr/simbad/sim-basic?Ident={encoded_name}"
    link_html    = (
        f"<a href='{url_simbad}' target='_blank' "
        f"style='color:#0056b3; font-weight:bold; text-decoration:none;'>{planet_name}</a>"
    )
    planet_links.append(link_html)
internal_data['planet_id'] = planet_links

# --- Discovery reference column ---
discovery_refs = []
for _, row in df.iterrows():
    raw_ref = row.get('discovery_ref', '')
    raw_ref = str(raw_ref) if pd.notna(raw_ref) else ''
    discovery_refs.append(format_reference_smart(raw_ref))
internal_data['discovery_ref'] = discovery_refs

# --- Detection method column ---
internal_data['method'] = df['Detection_Method_Full'].tolist()

# --- Habitability zone column (rendered as badge in-cell) ---
internal_data['hz_zone'] = [format_hz_zone_badge(value) for value in df.get('hz_zone', pd.Series(['No data'] * len(df)))]

# --- Planet category column (rendered as badge in-cell) ---
# This notebook consumes the category curated by Script 1.
internal_data['planet_category'] = [
    format_planet_category_badge(value)
    for value in df.get('planet_category', pd.Series(['Unknown'] * len(df)))
]

# --- Database links: Exoplanet.eu and NASA ---
eu_links, nasa_links = [], []
for planet_name in df['name']:
    planet_name = str(planet_name).strip()

    # Exoplanet.eu uses underscores in URLs
    name_eu  = planet_name.replace(" ", "_")
    url_eu   = f"http://exoplanet.eu/catalog/{name_eu}/"
    eu_links.append(
        f"<a href='{url_eu}' target='_blank' "
        f"style='color:#f59e0b; font-weight:bold; text-decoration:none;'>Eu</a>"
    )

    # NASA Exoplanet Archive uses percent-encoded names
    name_nasa = urllib.parse.quote(planet_name)
    url_nasa  = f"https://exoplanetarchive.ipac.caltech.edu/overview/{name_nasa}"
    nasa_links.append(
        f"<a href='{url_nasa}' target='_blank' "
        f"style='color:#0b3d91; font-weight:bold; text-decoration:none;'>NASA</a>"
    )

internal_data['eu_link']   = eu_links
internal_data['nasa_link'] = nasa_links

# --- Planet status column: colour-coded badges ---
status_html = []
for s in df.get('planet_status', pd.Series(['Confirmed'] * len(df))):
    s_str = str(s).strip().title()
    if s_str == 'Confirmed':
        status_html.append(
            "<span style='background-color:#d4edda; color:#155724; padding:3px 8px; "
            "border-radius:6px; font-weight:bold; font-size:0.85em;'>Confirmed</span>"
        )
    elif s_str == 'Candidate':
        status_html.append(
            "<span style='background-color:#ffe8a1; color:#856404; padding:3px 8px; "
            "border-radius:6px; font-weight:bold; font-size:0.85em;'>Candidate</span>"
        )
    elif s_str in ['Retracted', 'False Positive', 'Discarded', 'Controversial']:
        status_html.append(
            f"<span style='background-color:#f8d7da; color:#721c24; padding:3px 8px; "
            f"border-radius:6px; font-weight:bold; font-size:0.85em;'>{s_str}</span>"
        )
    else:
        status_html.append(s_str)
internal_data['status'] = status_html

# --- Coordinates (RA / Dec) ---
ra_values, dec_values = [], []
for _, row in df.iterrows():
    try:
        ra = float(row.get('ra', np.nan))
        ra_values.append(f"{ra:.4f}" if pd.notna(ra) else "")
    except Exception:
        ra_values.append("")

    try:
        dec = float(row.get('dec', np.nan))
        dec_values.append(f"{dec:.4f}" if pd.notna(dec) else "")
    except Exception:
        dec_values.append("")

internal_data['ra_val']  = ra_values
internal_data['dec_val'] = dec_values

# --- Spectral type ---
spt_values, spt_refs = [], []
for _, row in df.iterrows():
    spt_values.append(str(row.get('star_sp_type', '')) if pd.notna(row.get('star_sp_type')) else "")
    raw_ref = str(row.get('star_sp_type_ref', '')) if pd.notna(row.get('star_sp_type_ref')) else ""
    spt_refs.append(format_reference_smart(raw_ref))

internal_data['spt_val'] = spt_values
internal_data['spt_ref'] = spt_refs

# --- Physical parameter columns (value, error, reference) ---
for name, info in params.items():
    vals_col, errs_col, refs_col = [], [], []
    ref_col_name = info.get('ref', f"{info['val']}_ref")

    for _, row in df.iterrows():
        val   = row.get(info['val'], np.nan)
        e_min = row.get(info['min'], np.nan)
        e_max = row.get(info['max'], np.nan)
        ref_str = row.get(ref_col_name, "")

        is_min_mass = False

        # For planetary mass: fall back to M sin i if M is unavailable
        if name == 'M_p' and pd.isna(val):
            val     = row.get('mass_sini_earth',           np.nan)
            e_min   = row.get('mass_sini_earth_error_min', np.nan)
            e_max   = row.get('mass_sini_earth_error_max', np.nan)
            ref_str = row.get('mass_sini_ref', "")
            # Exoplanet.eu sometimes stores the reference in mass_ref instead
            if pd.isna(ref_str) or str(ref_str).strip() == "":
                ref_str = row.get('mass_ref', "")
            if pd.notna(val):
                is_min_mass = True

        # For planetary radius: use the Script 1 theoretical/fallback radius column.
        if name == 'R_p':
            radius_source = str(row.get('radius_earth_th_source', '')).strip()
            if pd.isna(val):
                # Legacy compatibility with older catalogues.
                val = row.get('radius_earth_hz', np.nan)
                radius_source = str(row.get('radius_earth_hz_source', '')).strip()

            if radius_source == 'estimated_from_mass':
                ref_str = 'This work'
            elif pd.isna(ref_str) or str(ref_str).strip() == '':
                ref_str = row.get('radius_ref', '')

        v_str, e_str = format_val_and_error(val, e_min, e_max)

        # Append M sin i label when using the minimum-mass column
        if is_min_mass and v_str != "":
            v_str += ' <span style="color:#999; font-size:0.8em; margin-left:2px;">sin <i>i</i></span>'

        # Reference overrides for specific parameters
        if name == 'T_0,p':
            ref_str = row.get('T0_val_ref', '')
        elif name == 'T_eq':
            ref_str = 'This work'
        elif name == '&#961;_star':
            if pd.isna(ref_str) or str(ref_str).strip() == '':
                ref_str = "-"

        if pd.isna(ref_str):
            ref_str = ""

        # Suppress reference if value is empty
        if v_str.strip() == "":
            ref_str = ""

        vals_col.append(v_str)
        errs_col.append(e_str)
        refs_col.append(format_reference_smart(ref_str))

    internal_data[f'{name}_val'] = vals_col
    internal_data[f'{name}_err'] = errs_col
    internal_data[f'{name}_ref'] = refs_col

Reading master catalogue: Exoplanets_SolarNeighbourhood_Catalogue.csv...


## Step 3 — Multi-Index DataFrame Construction

Column groups are assembled into a two-level MultiIndex: the top level is the parameter
name (with units), the bottom level is `Value`, `Error`, or `Ref`. RA and Dec are injected
immediately before the stellar radius group.

In [5]:
final_columns      = []
final_data_arrays  = []

header_id     = "ID<br><span style='font-size:0.8em; color:#666'>Planet</span>"
header_disc   = "Discovery<br><span style='font-size:0.8em; color:#666'>Ref</span>"
header_method = "Detection<br><span style='font-size:0.8em; color:#666'>Method</span>"
header_status = "Status<br><span style='font-size:0.8em; color:#666'>Planet</span>"
header_hz     = "HZ Zone"
header_cat    = "Planet Category"

final_columns.append((header_id,     "")); final_data_arrays.append(internal_data['planet_id'])
final_columns.append((header_disc,    "")); final_data_arrays.append(internal_data['discovery_ref'])
final_columns.append((header_method,  "")); final_data_arrays.append(internal_data['method'])
final_columns.append((header_status,   "")); final_data_arrays.append(internal_data['status'])
final_columns.append((header_hz,       "")); final_data_arrays.append(internal_data['hz_zone'])
final_columns.append((header_cat,      "")); final_data_arrays.append(internal_data['planet_category'])

for name, info in params.items():
    # RA and Dec are injected immediately before the stellar radius group
    if name == 'R_star':
        header_ra  = "RA<br><span style='font-size:0.85em; color:#666; font-weight:normal'>(deg)</span>"
        header_dec = "Dec<br><span style='font-size:0.85em; color:#666; font-weight:normal'>(deg)</span>"
        final_columns.append((header_ra,  "")); final_data_arrays.append(internal_data['ra_val'])
        final_columns.append((header_dec, "")); final_data_arrays.append(internal_data['dec_val'])

    unit_html   = f"<span style='font-size:0.85em; color:#666; font-weight:normal'>({info['unit']})</span>" if info['unit'] else ""
    header_html = f"{name}<br>{unit_html}"

    final_columns.append((header_html, 'Value')); final_data_arrays.append(internal_data[f'{name}_val'])
    final_columns.append((header_html, 'Error')); final_data_arrays.append(internal_data[f'{name}_err'])
    final_columns.append((header_html, 'Ref'));   final_data_arrays.append(internal_data[f'{name}_ref'])

header_spt = "Sp. Type"
final_columns.append((header_spt, "Value")); final_data_arrays.append(internal_data['spt_val'])
final_columns.append((header_spt, "Ref"));   final_data_arrays.append(internal_data['spt_ref'])

header_db = "Databases<br><span style='font-size:0.8em; color:#666'>Links</span>"
final_columns.append((header_db, "EU"));   final_data_arrays.append(internal_data['eu_link'])
final_columns.append((header_db, "NASA")); final_data_arrays.append(internal_data['nasa_link'])

df_final = pd.DataFrame(final_data_arrays).T
df_final.columns = pd.MultiIndex.from_tuples(final_columns)

## Step 4 — HTML Export

The styled DataFrame is embedded in a full HTML page with DataTables (search, sort, pagination)
and a synchronised top scrollbar for wide tables.

In [6]:
print("Generating HTML...")

# Locate the visible mass value column (M_p, Value) so sorting stays correct
# even when some rows display the "sin i" visual label.
mass_col_idx = None
for idx, (top, bottom) in enumerate(df_final.columns):
    if isinstance(top, str) and top.startswith("M_p<br") and bottom == "Value":
        mass_col_idx = idx
        break

if mass_col_idx is None:
    raise ValueError("Could not find M_p Value column index for DataTables sorting.")

html_table = (
    df_final.style
    .set_uuid("myTable")
    .set_table_attributes('class="display compact cell-border hover" style="width:100%"')
    .format(na_rep="")
    .hide(axis='index')
    .to_html(escape=False)
)

full_html = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>{table_title}</title>
    <link rel="stylesheet" type="text/css" href="https://cdn.datatables.net/1.13.6/css/jquery.dataTables.min.css">
    <style>
        body {{ font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 20px; background-color: #f4f4f4; }}
        h2 {{ text-align: center; color: #333; margin-bottom: 14px; font-weight: 600; }}
        .table-container {{ width: 95%; margin: 0 auto; background-color: white; padding: 20px; border-radius: 8px; box-shadow: 0 0 15px rgba(0,0,0,0.1); }}
        .sticky-scrollbar-wrapper {{ position: sticky; bottom: 0; width: 100%; overflow-x: auto; overflow-y: hidden; background: white; border-top: 1px solid #ddd; z-index: 100; }}
        .sticky-scrollbar-content {{ height: 1px; }}
        table.dataTable thead th {{ text-align: center; background-color: #2c3e50; color: white; font-size: 0.9em; vertical-align: bottom; border-bottom: 2px solid #1a252f; }}
        table.dataTable tbody td {{ font-size: 0.9em; text-align: center; vertical-align: middle; }}
        a {{ text-decoration: none; color: #007bff; transition: 0.2s; }}
        a:hover {{ text-decoration: underline; color: #0056b3; }}
    </style>
</head>
<body>
    <h2>{table_title}</h2>
    <div class="table-container">
        {html_table}
        <div id="stickyScroll" class="sticky-scrollbar-wrapper">
            <div id="stickyScrollContent" class="sticky-scrollbar-content"></div>
        </div>
    </div>
    <script src="https://code.jquery.com/jquery-3.7.0.min.js"></script>
    <script src="https://cdn.datatables.net/1.13.6/js/jquery.dataTables.min.js"></script>
    <script>
        $(document).ready(function() {{
            var table = $('#T_myTable').DataTable({{
                "paging": true, "pageLength": 20, "lengthMenu": [[10, 20, 50, -1], [10, 20, 50, "All"]],
                "ordering": true, "info": true, "searching": true, "scrollX": true,
                "columnDefs": [
                    {{ "type": "html-num", "targets": "_all" }},
                    {{
                        "targets": [{mass_col_idx}],
                        "render": function(data, type, row) {{
                            if (type === 'sort' || type === 'type') {{
                                var div = document.createElement('div');
                                div.innerHTML = data || '';
                                var text = (div.textContent || div.innerText || '').trim();

                                // Keep visual label "sin i" in display, but ignore it for numeric sorting.
                                text = text.replace(/sin\s*i/gi, '').trim();

                                // Accept optional limit markers and scientific notation.
                                var m = text.match(/^[<>~]?\s*[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?/);
                                if (!m) return -Infinity;

                                var numText = m[0].replace(/[<>~\s]/g, '');
                                var num = parseFloat(numText);
                                return Number.isNaN(num) ? -Infinity : num;
                            }}
                            return data;
                        }}
                    }}
                ]
            }});
            var stickyScroll = $('#stickyScroll');
            var stickyContent = $('#stickyScrollContent');
            var tableScroll   = $('.dataTables_scrollBody');

            function updateScrollWidth() {{ stickyContent.width($('#T_myTable').width()); }}
            stickyScroll.on('scroll', function() {{ tableScroll.scrollLeft($(this).scrollLeft()); }});
            tableScroll.on('scroll', function() {{ stickyScroll.scrollLeft($(this).scrollLeft()); }});
            updateScrollWidth();
            $(window).resize(updateScrollWidth);
            table.on('draw', updateScrollWidth);
        }});
    </script>
</body>
</html>
"""

with open(file_output, 'w', encoding='utf-8') as f:
    f.write(full_html)

print(f"Done. Open '{file_output}' in your browser.")

Generating HTML...
Done. Open 'exoplanets_catalogue.html' in your browser.


## Output Verification

In [7]:
print(f"Table shape: {df_final.shape[0]} rows × {df_final.shape[1]} columns")
print(f"Output written to: {file_output}\n")

print("Detection method breakdown:")
print(df['Detection_Method_Full'].value_counts().to_string())

print("\nPlanet status breakdown:")
print(df['planet_status'].value_counts().to_string())

Table shape: 130 rows × 54 columns
Output written to: exoplanets_catalogue.html

Detection method breakdown:
Detection_Method_Full
RV                 112
Transit + RV        12
RV + Astrometry      4
TTV                  1
Astrometry           1

Planet status breakdown:
planet_status
Confirmed    104
Candidate     19
Discarded      7
